In [ ]:
# Установка необходимых библиотек (если ещё не установлены)
!pip install catboost xgboost -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==================== 1. ЗАГРУЗКА И ПРЕДОБРАБОТКА ====================
df = pd.read_csv('SUBMISSION_DATA.csv', sep=';')
df = df.drop(columns=['ts_update']).dropna()
df['datetime'] = pd.to_datetime(df['ts_create'], unit='s')
df = df.drop(columns=['ts_create'])

# Оставляем только сенсор 1
df = df[df['sensor_id'] == 1.0].copy()
df = df.sort_values('datetime').reset_index(drop=True)

# Усреднение при дублях одного дня
df = df.groupby('datetime', as_index=False)['value'].mean()
df = df.sort_values('datetime').reset_index(drop=True)

# Переход к ежедневной частоте и интерполяция коротких пропусков (до 2 дней)
df = df.set_index('datetime').asfreq('D')
print(f'Пропусков до интерполяции: {df["value"].isna().sum()}')
df['value'] = df['value'].interpolate(method='linear', limit=2)
df = df.reset_index()

# ==================== 2. СОЗДАНИЕ ПРИЗНАКОВ ====================
lag_steps = [1, 3, 6, 12, 24, 48]
for lag in lag_steps:
    df[f'lag_{lag}'] = df['value'].shift(lag)

rolling_windows = [3, 6, 12, 24, 48]
for w in rolling_windows:
    df[f'roll_mean_{w}'] = df['value'].rolling(window=w, min_periods=w).mean()
    df[f'roll_std_{w}'] = df['value'].rolling(window=w, min_periods=w).std()

feature_cols = [f'lag_{lag}' for lag in lag_steps] + \
               [f'roll_mean_{w}' for w in rolling_windows] + \
               [f'roll_std_{w}' for w in rolling_windows]

# Календарные признаки
df['day_of_week'] = df['datetime'].dt.dayofweek
df['day_of_year'] = df['datetime'].dt.dayofyear
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

calendar_features = ['day_of_week', 'month', 'day', 'is_weekend',
                     'day_of_year_sin', 'day_of_year_cos']
all_features = feature_cols + calendar_features

# ==================== 3. ЦЕЛЕВЫЕ ПЕРЕМЕННЫЕ (3 ГОРИЗОНТА) ====================
df['target_1d'] = df['value'].shift(-1)   # следующий день
df['target_7d'] = df['value'].shift(-7)   # через неделю
df['target_30d'] = df['value'].shift(-30) # через месяц

# Удаляем строки с пропусками в признаках или целевых переменных
df_model = df.dropna(subset=all_features + ['target_1d', 'target_7d', 'target_30d']).copy().reset_index(drop=True)
print(f'Размер выборки после удаления NaN: {df_model.shape}')

# ==================== 4. WALK-FORWARD РАЗБИЕНИЕ ====================
min_train_days = 365
test_days = 30
step_days = 30

folds = []
start_time = df_model['datetime'].min()
end_time = df_model['datetime'].max()
train_end = start_time + pd.Timedelta(days=min_train_days)

while train_end + pd.Timedelta(days=test_days) <= end_time:
    test_start = train_end
    test_end = test_start + pd.Timedelta(days=test_days)
    train_df = df_model[df_model['datetime'] < train_end].copy()
    test_df = df_model[(df_model['datetime'] >= test_start) & (df_model['datetime'] < test_end)].copy()
    if len(train_df) > 0 and len(test_df) > 0:
        folds.append((train_df, test_df))
    train_end += pd.Timedelta(days=step_days)

print(f'Сформировано фолдов: {len(folds)}')

# ==================== 5. ФУНКЦИЯ ДЛЯ АВТОРСКОЙ МЕТРИКИ ====================
def cosine_risk_similarity(pred_matrix, risk_vector):
    """
    pred_matrix : np.ndarray, shape (n_samples, 3) – прогнозы на 1,7,30 дней
    risk_vector : np.ndarray, shape (3,) – вектор риска (например, 95-й перцентиль)
    Возвращает среднее косинусное сходство между каждым вектором прогноза и вектором риска.
    """
    norms = np.linalg.norm(pred_matrix, axis=1)
    norms[norms == 0] = 1  # защита от деления на ноль
    cos_sim = np.dot(pred_matrix, risk_vector) / (norms * np.linalg.norm(risk_vector))
    return np.mean(cos_sim)

# ==================== 6. ЦИКЛ ПО ФОЛДАМ ====================
results = {
    'fold': [],
    'catboost_mae': [], 'catboost_rmse': [], 'catboost_cosine': [],
    'xgboost_mae': [], 'xgboost_rmse': [], 'xgboost_cosine': [],
    'rf_mae': [], 'rf_rmse': [], 'rf_cosine': []
}

for fold_idx, (train_df, test_df) in enumerate(folds, 1):
    print(f'\nОбработка фолда {fold_idx}...')
    X_train = train_df[all_features]
    X_test = test_df[all_features]

    # Словари для хранения прогнозов
    preds_cb, preds_xgb, preds_rf = {}, {}, {}

    for horizon, target_col in zip(['1d', '7d', '30d'], ['target_1d', 'target_7d', 'target_30d']):
        y_train = train_df[target_col]
        y_test = test_df[target_col]

        # CatBoost
        cb = CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05,
                                loss_function='RMSE', verbose=False)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
        preds_cb[horizon] = cb.predict(X_test)

        # XGBoost
        xgb = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                            objective='reg:squarederror', random_state=42)
        xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        preds_xgb[horizon] = xgb.predict(X_test)

        # RandomForest
        rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
        rf.fit(X_train, y_train)
        preds_rf[horizon] = rf.predict(X_test)

    # Матрицы прогнозов (n_test, 3)
    pred_matrix_cb = np.column_stack([preds_cb['1d'], preds_cb['7d'], preds_cb['30d']])
    pred_matrix_xgb = np.column_stack([preds_xgb['1d'], preds_xgb['7d'], preds_xgb['30d']])
    pred_matrix_rf = np.column_stack([preds_rf['1d'], preds_rf['7d'], preds_rf['30d']])

    # Вектор риска по тренировочным данным (95-й перцентиль каждого горизонта)
    risk_vector = np.array([
        train_df['target_1d'].quantile(0.95),
        train_df['target_7d'].quantile(0.95),
        train_df['target_30d'].quantile(0.95)
    ])

    # Косинусная метрика
    cos_cb = cosine_risk_similarity(pred_matrix_cb, risk_vector)
    cos_xgb = cosine_risk_similarity(pred_matrix_xgb, risk_vector)
    cos_rf = cosine_risk_similarity(pred_matrix_rf, risk_vector)

    # Стандартные метрики для горизонта 7 дней
    y_test_7d = test_df['target_7d']
    results['fold'].append(fold_idx)
    results['catboost_mae'].append(mean_absolute_error(y_test_7d, preds_cb['7d']))
    results['catboost_rmse'].append(np.sqrt(mean_squared_error(y_test_7d, preds_cb['7d'])))
    results['catboost_cosine'].append(cos_cb)

    results['xgboost_mae'].append(mean_absolute_error(y_test_7d, preds_xgb['7d']))
    results['xgboost_rmse'].append(np.sqrt(mean_squared_error(y_test_7d, preds_xgb['7d'])))
    results['xgboost_cosine'].append(cos_xgb)

    results['rf_mae'].append(mean_absolute_error(y_test_7d, preds_rf['7d']))
    results['rf_rmse'].append(np.sqrt(mean_squared_error(y_test_7d, preds_rf['7d'])))
    results['rf_cosine'].append(cos_rf)

# ==================== 7. ИТОГОВЫЕ РЕЗУЛЬТАТЫ ====================
results_df = pd.DataFrame(results)
print('\nСводная таблица результатов по фолдам:')
display(results_df)

print('\nСредние метрики по фолдам:')
for model in ['catboost', 'xgboost', 'rf']:
    print(f"{model.upper()}: MAE(7d) = {results_df[f'{model}_mae'].mean():.4f}, "
          f"RMSE(7d) = {results_df[f'{model}_rmse'].mean():.4f}, "
          f"CosineRisk = {results_df[f'{model}_cosine'].mean():.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00
Пропусков до интерполяции: 0
Размер выборки после удаления NaN: (2114, 28)
Сформировано фолдов: 58

Обработка фолда 1...

Обработка фолда 2...

Обработка фолда 3...

Обработка фолда 4...

Обработка фолда 5...

Обработка фолда 6...

Обработка фолда 7...

Обработка фолда 8...

Обработка фолда 9...

Обработка фолда 10...

Обработка фолда 11...

Обработка фолда 12...

Обработка фолда 13...

Обработка фолда 14...

Обработка фолда 15...

Обработка фолда 16...

Обработка фолда 17...

Обработка фолда 18...

Обработка фолда 19...

Обработка фолда 20...

Обработка фолда 21...

Обработка фолда 22...

Обработка фолда 23...

Обработка фолда 24...

Обработка фолда 25...

Обработка фолда 26...

Обработка фолда 27...

Обработка фолда 28...

Обработка фолда 29...

Обработка фолда 30...

Обработка фолда 31...

Обработка фолда 32...

Обработка фолда 33...

Обработка фолда 34...

Обработка фолда 35...

Обработка фолда 36...

Обр

,fold,catboost_mae,catboost_rmse,catboost_cosine,xgboost_mae,xgboost_rmse,xgboost_cosine,rf_mae,rf_rmse,rf_cosine
0,1,2.188363,2.332546,0.996727,23.064851,24.440671,0.973314,25.479242,27.027753,0.970278
1,2,2.933705,3.218364,0.992278,7.845742,8.470354,0.990990,7.516487,8.195680,0.987940
2,3,1.358044,1.629284,0.997458,3.340519,4.302971,0.995046,3.601026,4.236989,0.994880
3,4,2.793301,2.958036,0.998398,3.659513,3.776075,0.996464,4.453030,4.654575,0.995327
4,5,1.482247,1.786277,0.997276,1.840069,2.165613,0.998962,2.254385,2.654635,0.998217
5,6,1.142156,1.380454,0.999590,1.444713,1.750067,0.997979,1.659501,1.979190,0.997810
6,7,1.330247,1.585222,0.997181,1.128126,1.415956,0.996465,1.220122,1.449894,0.995895
7,8,1.824037,2.150048,0.999323,3.070049,3.947450,0.994955,2.971346,3.795808,0.996908
8,9,1.084242,1.390572,0.998423,4.938301,7.081070,0.979546,2.593862,3.560587,0.994794
9,10,1.150088,1.246451,0.998692,1.335200,1.518243,0.998540,1.602433,1.766346,0.997516



Средние метрики по фолдам:
CATBOOST: MAE(7d) = 7.0833, RMSE(7d) = 8.3164, CosineRisk = 0.9946
XGBOOST: MAE(7d) = 9.3783, RMSE(7d) = 10.9730, CosineRisk = 0.9889
RF: MAE(7d) = 9.4376, RMSE(7d) = 10.9432, CosineRisk = 0.9897
